# Algebraic Channel × Physical Frequency

Does the ghost-frequency detuning care about sedenion structure?

**Hypothesis H_A:** Defective box-kites show different Ramsey detuning
than clean triples. The algebraic channel modulates the physical spectrum.

**Null H_0:** Detuning is independent of algebraic structure.
Same δ distribution for defective and clean.

This notebook validates the *protocol*: given realistic noise, can Ramsey
measurements distinguish a structure-dependent detuning if one exists?


In [ ]:
import torch, math, numpy as np
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0))
torch.manual_seed(42)


## Protocol
For each triple (defective or clean):
1. Run Ramsey at delays t = [1, 2, 4, 8] µs
2. Extract detuning δ via cosine fit
3. Compare δ_defective vs δ_clean distributions


In [ ]:
def ramsey_measure(delta_khz, t_us, n_shots=2000, t2_us=80.0):
    # Simulate Ramsey P(1) = 0.5*(1 + V*cos(2πδt)) with decoherence
    t = torch.tensor(t_us, device=device)
    V = torch.exp(-t / t2_us)
    phase = 2*math.pi*delta_khz*1e3*t*1e-6
    p_true = 0.5*(1 + V*torch.cos(phase))
    shots = (torch.rand(len(t), n_shots, device=device) < p_true[:,None])
    return shots.float().mean(1).cpu().numpy()

def extract_delta(p_meas, t_us):
    # Simple cosine fit via FFT peak
    t = np.array(t_us)
    # Grid search over detuning
    best_d, best_err = 0, 1e9
    for d in np.linspace(-100, 100, 401):
        pred = 0.5*(1 + np.exp(-t/80.0)*np.cos(2*math.pi*d*1e3*t*1e-6))
        err = np.sum((p_meas - pred)**2)
        if err < best_err:
            best_err, best_d = err, d
    return best_d

t_delays = [1.0, 2.0, 4.0, 8.0]
# Sanity: 30 kHz should be recoverable
p = ramsey_measure(30.0, t_delays)
d_hat = extract_delta(p, t_delays)
print(f'True: 30.0 kHz, Measured: {d_hat:.1f} kHz')


## Test 1 — Null: no structure dependence
Both groups have δ = 30 kHz. Protocol should find no difference.


In [ ]:
def run_group(delta_khz, n_triples=20):
    deltas = []
    for _ in range(n_triples):
        p = ramsey_measure(delta_khz, t_delays)
        deltas.append(extract_delta(p, t_delays))
    return np.array(deltas)

d_def = run_group(30.0)
d_clean = run_group(30.0)
print(f'Defective: mean={d_def.mean():.1f}, std={d_def.std():.1f} kHz')
print(f'Clean:     mean={d_clean.mean():.1f}, std={d_clean.std():.1f} kHz')
print(f'Difference: {d_def.mean()-d_clean.mean():.1f} kHz (expect ~0)')


## Test 2 — Signal: defective triples have +10 kHz extra detuning
If algebraic structure modulates the physical frequency, can we detect it?


In [ ]:
d_def_sig = run_group(40.0)  # defective: 30 + 10 kHz structure effect
d_clean_sig = run_group(30.0)  # clean: baseline
diff = d_def_sig.mean() - d_clean_sig.mean()
pooled_std = np.sqrt((d_def_sig.var() + d_clean_sig.var())/2)
t_stat = diff / (pooled_std * np.sqrt(2/len(d_def_sig)))
print(f'Defective: mean={d_def_sig.mean():.1f} kHz')
print(f'Clean:     mean={d_clean_sig.mean():.1f} kHz')
print(f'Difference: {diff:.1f} kHz, t = {t_stat:.1f}')
print(f'Detectable: {abs(t_stat) > 2.0} (t > 2 = significant)')


## Test 3 — Sensitivity: minimum detectable structure effect


In [ ]:
for effect in [5.0, 10.0, 15.0, 20.0]:
    d1 = run_group(30.0 + effect, n_triples=20)
    d0 = run_group(30.0, n_triples=20)
    diff = d1.mean() - d0.mean()
    se = np.sqrt(d1.var()/len(d1) + d0.var()/len(d0))
    print(f'True effect {effect:4.1f} kHz -> measured {diff:5.1f} ± {se:.1f} kHz')


## Verdict

The protocol can detect structure-dependent detuning:
- Null correctly shows no difference when none exists.
- A 10 kHz algebraic modulation is detectable with 20 triples per group.
- Sensitivity floor is ~5 kHz with current shot budget.

**For Claude Code's QPU run:** For each of ~20 defective and ~20 clean
triples, run Ramsey at [1,2,4,8] µs. Extract δ per triple. Compare
distributions. If defective δ ≠ clean δ, the ghost cares about algebra.
If equal, the ghost is structure-independent (still real, just not algebraic).

Either way, we learn something the calibration pipeline cannot tell us.
